In [5]:
import numpy as np
import pandas as pd

lambda0 = 10
lambda1 = 20

x_pois = np.array([
    14, 10, 6, 10, 9, 7, 17, 9, 13, 10,
    22, 21, 15, 25, 27, 26, 20, 26, 26, 17,
    23, 16, 18, 22, 15, 23, 19, 15, 18, 13
])

k_pois = (lambda1 - lambda0) / (np.log(lambda1) - np.log(lambda0))

def estimate_arl0_pois_fast(h, M=10000, seed=611211106):
    rng = np.random.default_rng(seed)

    C = np.zeros(M)
    RL = np.zeros(M, dtype=int)
    alive = np.ones(M, dtype=bool)

    n = 0
    while alive.any():
        n += 1

        idx = np.where(alive)[0]
        x = rng.poisson(lambda0, size=len(idx))

        C[idx] = np.maximum(0, C[idx] + x - k_pois)

        signal_idx = idx[C[idx] > h]
        RL[signal_idx] = n
        alive[signal_idx] = False

    return RL.mean()

def search_h_pois_fast(target_arl0=200, rho=0.5, M_search=5000, M_final=100000, seed=611211106):
    h_low = 0
    h_high = 50

    while True:
        h_mid = (h_low + h_high) / 2
        arl_mid = estimate_arl0_pois_fast(h_mid, M=M_search, seed=seed)

        if abs(arl_mid - target_arl0) <= rho:
            break

        if arl_mid < target_arl0:
            h_low = h_mid
        else:
            h_high = h_mid

    arl_final = estimate_arl0_pois_fast(h_mid, M=M_final, seed=seed)
    return h_mid, arl_final

h_pois, arl0_pois = search_h_pois_fast(target_arl0=200, rho=0.5)

n_pois = len(x_pois)
C_pois = np.zeros(n_pois)

for i in range(n_pois):
    prev = 0 if i == 0 else C_pois[i - 1]
    C_pois[i] = max(0, prev + x_pois[i] - k_pois)

signal_pois = C_pois > h_pois
first_signal_pois = np.where(signal_pois)[0][0] + 1 if np.any(signal_pois) else None

df_pois = pd.DataFrame({
    "n": np.arange(1, n_pois + 1),
    "X": x_pois,
    "C_plus": C_pois,
    "signal": signal_pois
})

print(f"k = {k_pois:.3f}")
print(f"h = {h_pois:.3f}")
print(f"Estimated ARL0 = {arl0_pois:.3f}")
print(f"First signal = {first_signal_pois}")
print(df_pois)

KeyboardInterrupt: 